# 1. Problem Statement

This notebook addresses the **Classification Track** of the ML Capstone Project (Review 1, Part A).

**Dataset:** Student records from a higher-education institution, containing demographic, socio-economic,
and academic-performance information collected at the time of enrollment and at the end of the first and
second semesters.

**Task:** Predict the `Target` column, which records the student's academic outcome at the end of the
course, using the other 36 columns as input features. This is a **multi-class classification** problem
with three possible outcomes: `Dropout`, `Enrolled`, and `Graduate`.

**Why the target column is not ambiguous:** the dataset ships with a column literally named `Target`
containing exactly three categorical outcome labels (`Dropout`, `Enrolled`, `Graduate`), while every other
column is a numerically-coded predictor (demographics, admission data, or semester performance). There is
no other plausible label column, so `Target` was used directly without needing to ask for clarification.

# 2. Dataset Description

- **Rows:** 4,424 students
- **Columns:** 37 (36 features + 1 target)
- **Target:** `Target` — three classes: `Dropout`, `Enrolled`, `Graduate`
- **Feature groups:**
  - Demographic: `Marital status`, `Nacionality`, `Gender`, `Age at enrollment`, `International`
  - Socio-economic: `Displaced`, `Educational special needs`, `Debtor`, `Tuition fees up to date`,
    `Scholarship holder`, `Mother's/Father's qualification`, `Mother's/Father's occupation`,
    `Unemployment rate`, `Inflation rate`, `GDP`
  - Academic/admission: `Application mode`, `Application order`, `Course`,
    `Daytime/evening attendance`, `Previous qualification` (+ grade), `Admission grade`
  - Semester performance (1st and 2nd semester): credited / enrolled / evaluations / approved
    curricular units, and semester grade

This description is based directly on the columns present in the dataset audit below — no additional
columns were invented.

In [1]:
# Core libraries
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & modelling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.autolayout"] = True

print("Libraries imported successfully.")

Libraries imported successfully.


In [8]:
# Load the actual classification dataset
df = pd.read_csv("../data/data.csv", sep=";", encoding="utf-8-sig")

# Clean stray tab characters and whitespace from column names
df.columns = [c.replace("\t", "").strip() for c in df.columns]

# Display first few rows
df.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


# 5 Dataset Audit

In [9]:
print("Shape (rows, columns):", df.shape)

Shape (rows, columns): (4424, 37)


In [10]:
df.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification                   

In [12]:
print("Data types:")
print(df.dtypes.value_counts())
print()
print("Missing values per column (top 10 shown):")
print(df.isnull().sum().sort_values(ascending=False).head(10))
print()
print("Total missing values in dataset:", df.isnull().sum().sum())

Data types:
int64      29
float64     7
object      1
Name: count, dtype: int64

Missing values per column (top 10 shown):
Marital status                    0
Application mode                  0
Application order                 0
Course                            0
Daytime/evening attendance        0
Previous qualification            0
Previous qualification (grade)    0
Nacionality                       0
Mother's qualification            0
Father's qualification            0
dtype: int64

Total missing values in dataset: 0


In [13]:
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 0


In [14]:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", categorical_cols)
print()
print("Numerical columns (%d):" % len(numerical_cols))
print(numerical_cols)

Categorical columns: ['Target']

Numerical columns (36):
['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Previous qualification (grade)', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Admission grade', 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Unemploymen

In [15]:
print("Target class distribution (counts):")
print(df["Target"].value_counts())
print()
print("Target class distribution (%):")
print((df["Target"].value_counts(normalize=True) * 100).round(2))

Target class distribution (counts):
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Target class distribution (%):
Target
Graduate    49.93
Dropout     32.12
Enrolled    17.95
Name: proportion, dtype: float64
